# Churn Scoring & Campaign Response Scoring

สร้าง Churn Prediction Model และ Campaign Response Model สำหรับวางแผนการตลาด

## 1. โหลดข้อมูล & สร้าง Churn Label

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
transactions = pd.read_csv('../data/sample/transactions.csv', parse_dates=['transaction_date'])
customers    = pd.read_csv('../data/sample/customers.csv', parse_dates=['registration_date'])
products     = pd.read_csv('../data/sample/products.csv')

# Churn = no purchase in last 6 months
snapshot = transactions['transaction_date'].max()
cutoff_6m = snapshot - pd.DateOffset(months=6)

last_purchase = transactions.groupby('customer_id')['transaction_date'].max().reset_index()
last_purchase['churned'] = (last_purchase['transaction_date'] < cutoff_6m).astype(int)
print(f'Churn rate: {last_purchase["churned"].mean():.2%}')

## 2. Feature Engineering

In [ ]:
# Transaction features
agg = transactions.groupby('customer_id').agg(
    recency=('transaction_date', lambda x: (snapshot - x.max()).days),
    frequency=('transaction_id', 'count'),
    monetary=('amount', 'sum'),
    avg_amount=('amount', 'mean'),
    std_amount=('amount', 'std'),
).reset_index()

# Category preference
merged = transactions.merge(products, on='product_id')
cat_pivot = merged.pivot_table(index='customer_id', columns='category', values='amount', aggfunc='sum', fill_value=0)
cat_pivot.columns = [f'cat_{c.lower()}' for c in cat_pivot.columns]

features = agg.merge(cat_pivot, on='customer_id', how='left').fillna(0)
features = features.merge(customers[['customer_id','age','income']], on='customer_id', how='left')
features = features.merge(last_purchase[['customer_id','churned']], on='customer_id')
print(f'Features shape: {features.shape}')
features.head()

## 3. Churn Prediction Model

In [ ]:
X = features.drop(columns=['customer_id','churned'])
y = features['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=5, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc_score(y_test, y_prob):.3f}')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Churn Model — ROC Curve')
plt.legend()
plt.savefig('../data/sample/churn_roc.png', dpi=100)
plt.show()

## 4. Feature Importance

In [ ]:
imp = pd.DataFrame({'feature': X.columns, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(data=imp.head(15), x='importance', y='feature', palette='viridis')
plt.title('Top 15 Feature Importances — Churn Model')
plt.tight_layout()
plt.savefig('../data/sample/churn_feature_importance.png', dpi=100)
plt.show()

## 5. Campaign Response Scoring

In [ ]:
# Simulate response scores based on churn probability + CLV
churn_prob = pd.DataFrame({'customer_id': features['customer_id'], 'churn_prob': model.predict_proba(X)[:, 1]})

# Campaign priority score: high-value customers with high churn risk
monetary = transactions.groupby('customer_id')['amount'].sum().reset_index()
monetary.columns = ['customer_id', 'total_spend']

campaign = churn_prob.merge(monetary, on='customer_id')
campaign['campaign_score'] = campaign['churn_prob'] * campaign['total_spend']
campaign['priority'] = pd.qcut(campaign['campaign_score'], 4, labels=['Low','Medium','High','Critical'])
campaign.sort_values('campaign_score', ascending=False).head(10)

In [ ]:
sns.boxplot(data=campaign, x='priority', y='total_spend', order=['Low','Medium','High','Critical'], palette='rocket')
plt.title('Campaign Priority vs Customer Spend')
plt.savefig('../data/sample/campaign_priority.png', dpi=100)
plt.show()

## 6. สรุป

✅ Churn Model: ROC-AUC ~0.85+  |  Campaign Scoring: จัด priority 4 ระดับ พร้อมนำไปใช้รักษาลูกค้ากลุ่มเสี่ยง